In [1]:
%matplotlib inline
import os
import re
import itertools
from itertools import combinations
import glob
import pickle
import argparse

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import multiprocess as mp
import numpy as np
import pandas as pd
import bioframe
#import cooltools
import cooler
from matplotlib import gridspec
from matplotlib.gridspec import GridSpec
import matplotlib.colors as colors
from matplotlib.colors import ListedColormap
import gzip
from collections import OrderedDict

In [2]:
import plotly.io as pio
import plotly.express as px
# Check the current configuration settings
current_config = pio.renderers
print("Current Plotly Configuration:")
print(current_config)

# Update configuration settings (example)
pio.renderers.default = 'notebook'  # Set the default renderer to notebook

# Print the updated configuration
print("\nUpdated Plotly Configuration:")
print(pio.renderers)

Current Plotly Configuration:
Renderers configuration
-----------------------
    Default renderer: 'plotly_mimetype+notebook'
    Available renderers:
        ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
         'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
         'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
         'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
         'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']


Updated Plotly Configuration:
Renderers configuration
-----------------------
    Default renderer: 'notebook'
    Available renderers:
        ['plotly_mimetype', 'jupyterlab', 'nteract', 'vscode',
         'notebook', 'notebook_connected', 'kaggle', 'azure', 'colab',
         'cocalc', 'databricks', 'json', 'png', 'jpeg', 'jpg', 'svg',
         'pdf', 'browser', 'firefox', 'chrome', 'chromium', 'iframe',
         'iframe_connected', 'sphinx_gallery', 'sphinx_gallery_png']



In [ ]:
#ES single cell stages (Plate6):
path = "/home/xiangru.huo-umw/scHiC/4DN_0829_2023/H1-plate5-scHiC-results/pairs_library"

condition = []
for i in range(1, 97):
    cond = f"scHiC_cell{i}.hg38.nodups.pairs.gz"
    condition.append(cond)
    
import os
os.chdir(path)
os.getcwd()

pairs_header = ['read_id', 'chrom1', 'pos1', 'chrom2', 'pos2', 'strand1', 'strand2', 'pair_type', 'mapq1', 'mapq2']

ES6_dfs = {}
ES_Num_Contacts6 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    ES_Num_Contacts6 = ES_Num_Contacts6+[Num_Cont]
    ES6_dfs[cond] = {'df': df}

    
#ES single cell stages (Plate8):
path = "/home/xiangru.huo-umw/scHiC/4DN_0829_2023/H1-plate7-scHiC-results/pairs_library"
    
import os
os.chdir(path)
os.getcwd()

pairs_header = ['read_id', 'chrom1', 'pos1', 'chrom2', 'pos2', 'strand1', 'strand2', 'pair_type', 'mapq1', 'mapq2']

ES8_dfs = {}
ES_Num_Contacts8 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    ES_Num_Contacts8 = ES_Num_Contacts8+[Num_Cont]
    ES8_dfs[cond] = {'df': df}
    
    
    
#combine two plates
ES_combined_dfs = {}

for cond in condition:
    es6_df = ES6_dfs[cond]['df']
    es8_df = ES8_dfs[cond]['df']
    
    ES_combined_dfs[cond] = {'plate6_': es6_df, 'plate8_': es8_df}
    
ES_Num_Contacts = {}
ES_Num_Contacts = {'plate6_': ES_Num_Contacts6, 'plate8_': ES_Num_Contacts8}

In [ ]:
import os
import gzip
import pandas as pd

output_file = "/home/xiangru.huo-umw/scHiC/GEO_processed_data/ES_plate_based.pairs.gz"

pairs_header = [
    "read_id", "chrom1", "pos1", "chrom2", "pos2",
    "strand1", "strand2", "pair_type", "mapq1", "mapq2"
]

with gzip.open(output_file, "wt") as out:

    # Optional header describing extra columns
    out.write("#combined ES plate-based scHi-C pairs\n")
    out.write("#columns: cell_id\tplate_id\t" + "\t".join(pairs_header) + "\n")

    for cond in condition:

        cell_id = cond.replace(".hg38.nodups.pairs.gz", "")

        for plate_name, df in [
            ("plate6", ES6_dfs[cond]["df"]),
            ("plate8", ES8_dfs[cond]["df"])
        ]:

            df_out = df.copy()

            # insert metadata columns at the beginning
            df_out.insert(0, "plate_id", plate_name)
            df_out.insert(0, "cell_id", cell_id)

            df_out.to_csv(
                out,
                sep="\t",
                index=False,
                header=False
            )

print(f"Saved to:\n{output_file}")

In [5]:
#DE single cell stages (Plate6):
path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/DE-plate6-HiC-results/pairs_library/"

condition = []
for i in range(1, 97):
    cond = f"scHiC_cell{i}.hg38.nodups.pairs.gz"
    condition.append(cond)
    
import os
os.chdir(path)
os.getcwd()

pairs_header = ['read_id', 'chrom1', 'pos1', 'chrom2', 'pos2', 'strand1', 'strand2', 'pair_type', 'mapq1', 'mapq2']

DE6_dfs = {}
DE_Num_Contacts6 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    DE_Num_Contacts6 = DE_Num_Contacts6+[Num_Cont]
    DE6_dfs[cond] = {'df': df}
    
    
#DE single cell stages (Plate8):
path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/DE-plate8-HiC-results/pairs_library/"

os.chdir(path)
os.getcwd()

DE8_dfs = {}
DE_Num_Contacts8 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    DE_Num_Contacts8 = DE_Num_Contacts8+[Num_Cont]
    DE8_dfs[cond] = {'df': df}
    
    
#combine two plates
DE_combined_dfs = {}

# Combine DE6 and DE8 dataframes for each condition
for cond in condition:
    de6_df = DE6_dfs[cond]['df']
    de8_df = DE8_dfs[cond]['df']
    
    # Combine DE6 and DE8 dataframes using a new key (condition) in the combined dictionary
    DE_combined_dfs[cond] = {'plate6_': de6_df, 'plate8_': de8_df}
    
DE_Num_Contacts = {}
DE_Num_Contacts = {'plate6_': DE_Num_Contacts6, 'plate8_': DE_Num_Contacts8}

In [6]:
#HB single cell stages (Plate6):
path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/HB-plate6-HiC-results/pairs_library/"

condition = []
for i in range(1, 97):
    cond = f"scHiC_cell{i}.hg38.nodups.pairs.gz"
    condition.append(cond)
    
import os
os.chdir(path)
os.getcwd()

pairs_header = ['read_id', 'chrom1', 'pos1', 'chrom2', 'pos2', 'strand1', 'strand2', 'pair_type', 'mapq1', 'mapq2']

HB6_dfs = {}
HB_Num_Contacts6 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    HB_Num_Contacts6 = HB_Num_Contacts6+[Num_Cont]
    HB6_dfs[cond] = {'df': df}
    
    
#HB single cell stages (Plate8):
path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/HB-plate8-HiC-results/pairs_library/"

os.chdir(path)
os.getcwd()

HB8_dfs = {}
HB_Num_Contacts8 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    HB_Num_Contacts8 = HB_Num_Contacts8+[Num_Cont]
    HB8_dfs[cond] = {'df': df}
    
    
#combine two plates
HB_combined_dfs = {}

for cond in condition:
    hb6_df = HB6_dfs[cond]['df']
    hb8_df = HB8_dfs[cond]['df']
    
    HB_combined_dfs[cond] = {'plate6_': hb6_df, 'plate8_': hb8_df}
    
HB_Num_Contacts = {}
HB_Num_Contacts = {'plate6_': HB_Num_Contacts6, 'plate8_': HB_Num_Contacts8}

In [7]:
#iHep single cell stages (Plate1):
path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/iHep-plate1-HiC-results/pairs_library/"

condition = []
for i in range(1, 97):
    cond = f"scHiC_cell{i}.hg38.nodups.pairs.gz"
    condition.append(cond)
    
import os
os.chdir(path)
os.getcwd()

pairs_header = ['read_id', 'chrom1', 'pos1', 'chrom2', 'pos2', 'strand1', 'strand2', 'pair_type', 'mapq1', 'mapq2']

iHep6_dfs = {}
iHep_Num_Contacts6 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    iHep_Num_Contacts6 = iHep_Num_Contacts6+[Num_Cont]
    iHep6_dfs[cond] = {'df': df}
    
    
#iHep single cell stages (Plate3):
path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/iHep-plate3-HiC-results/pairs_library/"

os.chdir(path)
os.getcwd()

iHep8_dfs = {}
iHep_Num_Contacts8 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    iHep_Num_Contacts8 = iHep_Num_Contacts8+[Num_Cont]
    iHep8_dfs[cond] = {'df': df}
    
    
#combine two plates
iHep_combined_dfs = {}

for cond in condition:
    ihep6_df = iHep6_dfs[cond]['df']
    ihep8_df = iHep8_dfs[cond]['df']
    
    iHep_combined_dfs[cond] = {'plate6_': ihep6_df, 'plate8_': ihep8_df}
    
iHep_Num_Contacts = {}
iHep_Num_Contacts = {'plate6_': iHep_Num_Contacts6, 'plate8_': iHep_Num_Contacts8}

In [8]:
#mHep single cell stages (Plate1):
path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/mHep-plate5-HiC-results/pairs_library/"
#path = "/home/xiangru.huo-umw/scHiC/mapping2/mHep-plate5-HiC-results/pairs_library"

condition = []
for i in range(1, 97):
    cond = f"scHiC_cell{i}.hg38.nodups.pairs.gz"
    condition.append(cond)
    
import os
os.chdir(path)
os.getcwd()

pairs_header = ['read_id', 'chrom1', 'pos1', 'chrom2', 'pos2', 'strand1', 'strand2', 'pair_type', 'mapq1', 'mapq2']

mHep6_dfs = {}
mHep_Num_Contacts6 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    mHep_Num_Contacts6 = mHep_Num_Contacts6+[Num_Cont]
    mHep6_dfs[cond] = {'df': df}
    
    
#mHep single cell stages (Plate3):
path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/mHep-plate7-HiC-results/pairs_library/"
#path = "/home/xiangru.huo-umw/scHiC/4DN_08122023_HiC/mHep-plate6-HiC-results_w/pairs_library" #new batch data

os.chdir(path)
os.getcwd()

mHep8_dfs = {}
mHep_Num_Contacts8 = [ ]
for cond in condition:
    with gzip.open(cond, 'rt') as file:
        df = pd.read_csv(
            f"{path}/{cond}",
            sep="\t",
            header=None,
            comment="#",
            names=pairs_header,
            index_col=None
        )
    Num_Cont = len(df)
    mHep_Num_Contacts8 = mHep_Num_Contacts8+[Num_Cont]
    mHep8_dfs[cond] = {'df': df}
    
    
#combine two plates
mHep_combined_dfs = {}
for cond in condition:
    mhep6_df = mHep6_dfs[cond]['df']
    mhep8_df = mHep8_dfs[cond]['df']    
    mHep_combined_dfs[cond] = {'plate6_': mhep6_df, 'plate8_': mhep8_df}

mHep_Num_Contacts = {}
mHep_Num_Contacts = {'plate6_': mHep_Num_Contacts6, 'plate8_': mHep_Num_Contacts8}

In [9]:
#Define stages
stages = ['H1','ES', 'DE','HB','iHep','mHep']

#organize five stages together
stage_combined_dfs = {}
stage_combined_dfs['H1'] = ES_combined_dfs
stage_combined_dfs['ES'] = ES_combined_dfs
stage_combined_dfs['DE'] = DE_combined_dfs
stage_combined_dfs['HB'] = HB_combined_dfs
stage_combined_dfs['iHep'] = iHep_combined_dfs
stage_combined_dfs['mHep'] = mHep_combined_dfs

Contacts_number = {}
Contacts_number['H1'] = ES_Num_Contacts
Contacts_number['ES'] = ES_Num_Contacts
Contacts_number['DE'] = DE_Num_Contacts
Contacts_number['HB'] = HB_Num_Contacts
Contacts_number['iHep'] = iHep_Num_Contacts
Contacts_number['mHep'] = mHep_Num_Contacts

In [10]:
#caculate the distance and only preserve "chrom1 == chrom 2" (intra-chrom contacts)
stage_contacts_dfs = {stage: {cond: {} for cond in condition} for stage in stages}

for stage in stages:
    for cond in condition:
        for plate in ['plate6_','plate8_']:
            df = stage_combined_dfs[stage][cond][plate].copy()
            df = df[df['chrom1'] == df['chrom2']]
            df['dist'] = df['pos1']-df['pos2']
            df['Abs_Dist'] = abs(df['dist'].copy())
            stage_contacts_dfs[stage][cond][plate] = df

In [11]:
#caculate short-range (1.5Kb-2Mb) and mitotic band (2-12Mb) ratio per cell 
# Initialize dictionaries for ratios and contact counts
short_range_ratio = {stage: {cond: {plate: [] for plate in ['plate6_', 'plate8_']} for cond in condition} for stage in stages}
mitotic_range_ratio = {stage: {cond: {plate: [] for plate in ['plate6_', 'plate8_']} for cond in condition} for stage in stages}
Num_Contacts = {stage: {cond: {plate: [] for plate in ['plate6_', 'plate8_']} for cond in condition} for stage in stages}

# Calculate ratios and store them in dictionaries
for stage in stages:
    for plate in ['plate6_', 'plate8_']:
        i = 0
        for cond in condition:        
            df = stage_contacts_dfs[stage][cond][plate].copy()
            short_range_t = df[(df['Abs_Dist'] > 1500) & (df['Abs_Dist'] < 2000000)]
            mitotic_range_t = df[(df['Abs_Dist'] > 2000000) & (df['Abs_Dist'] < 12000000)]
            Total_incis_1k = df[df['Abs_Dist'] > 1500]
            Num_Con = Contacts_number[stage][plate][i]
            i = i + 1
            short_range_ratio_t = len(short_range_t) / (len(Total_incis_1k)+0.001) #imputation incase some cells has 0 contacts
            mitotic_range_ratio_t = len(mitotic_range_t) / (len(Total_incis_1k)+0.001) #imputation incase some cells has 0 contacts
            
            # Append ratios and contact counts to dictionaries
            short_range_ratio[stage][cond][plate].append(short_range_ratio_t)
            mitotic_range_ratio[stage][cond][plate].append(mitotic_range_ratio_t)
            Num_Contacts[stage][cond][plate].append(Num_Con)

In [12]:
# Organize ShortVSMitotic dataframe
ShortVSMitotic = []  
for stage in stages:
    for plate in ['plate6_', 'plate8_']:
        for cond in condition:        
            ShortVSMitotic_t = pd.DataFrame([short_range_ratio[stage][cond][plate], mitotic_range_ratio[stage][cond][plate], Num_Contacts[stage][cond][plate], [stage], [plate],[plate+stage]]).transpose()
            ShortVSMitotic_t.index = [cond]  # Convert cond to a list to match the shape
            ShortVSMitotic_t.columns = ["Short_ratio(1.5K-2Mb)", "Mitotic_ratio(2Mb-12Mb)", "Num_Contacts", "stage", "plate",'plate+stage']
            ShortVSMitotic.append(ShortVSMitotic_t)

# Combine all DataFrames into a single DataFrame
ShortVSMitotic_df = pd.concat(ShortVSMitotic)
ShortVSMitotic_df

,Short_ratio(1.5K-2Mb),Mitotic_ratio(2Mb-12Mb),Num_Contacts,stage,plate,plate+stage
scHiC_cell1.hg38.nodups.pairs.gz,0.820918,0.068146,3232,H1,plate6_,plate6_H1
scHiC_cell2.hg38.nodups.pairs.gz,0.859035,0.05514,30092,H1,plate6_,plate6_H1
scHiC_cell3.hg38.nodups.pairs.gz,0.749934,0.118303,46420,H1,plate6_,plate6_H1
scHiC_cell4.hg38.nodups.pairs.gz,0.842731,0.079626,13789,H1,plate6_,plate6_H1
scHiC_cell5.hg38.nodups.pairs.gz,0.837094,0.0765,8808,H1,plate6_,plate6_H1
...,...,...,...,...,...,...
scHiC_cell92.hg38.nodups.pairs.gz,0.714252,0.095234,2367,mHep,plate8_,plate8_mHep
scHiC_cell93.hg38.nodups.pairs.gz,0.710508,0.157891,2064,mHep,plate8_,plate8_mHep
scHiC_cell94.hg38.nodups.pairs.gz,0.798147,0.079629,6318,mHep,plate8_,plate8_mHep
scHiC_cell95.hg38.nodups.pairs.gz,0.726076,0.138741,41923,mHep,plate8_,plate8_mHep


In [18]:
#filter cells > 30K ,> 50K or > 100K contacts & plot the ratio of short VS mitotic contacts
df = ShortVSMitotic_df[ShortVSMitotic_df['Num_Contacts']>5000]
fig = px.scatter(df, x='Mitotic_ratio(2Mb-12Mb)', y='Short_ratio(1.5K-2Mb)', color='plate+stage',
                 title="Short VS mitotic contacts ratio per cell (>5K)")
fig.update_layout(
    autosize=False,
    width=500,
    height=500,
    margin=dict(l=30, r=30, t=30, b=20),
    plot_bgcolor='rgba(0,0,0,0)',  # Set background color to transparent
    paper_bgcolor='rgba(0,0,0,0)'  # Set paper background color to transparent
)

fig.update_layout(
    shapes=[
        dict(
            type='rect',
            xref='x',
            yref='y',
            x0=0,
            y0=0.2,
            x1=0.4,
            y1=1,
            line=dict(color='black', width=1)  # Set line color and width
        ),
        dict(
            type='rect',
            xref='x',
            yref='y',
            x0=0.02,
            y0=0.7,
            x1=0.15,
            y1=0.95,
            line=dict(color='blue', width=1)  # Add a second box with a different color (red)
        ),
        dict(
            type='rect',
            xref='x',
            yref='y',
            x0=0.25,
            y0=0.45,
            x1=0.38,
            y1=0.65,
            line=dict(color='red', width=1)  # Add a second box with a different color (red)
        )
    ]
)

fig.update_traces(
    marker=dict(
        size=5,  # Set the marker size
        opacity=0.8  # Set marker transparency (0 is fully transparent, 1 is opaque)
    )
)

pio.write_image(fig, "/home/xiangru.huo-umw/scHiC/scripts/plot/5stage_Short_VS_mitotic_contacts_ratio_per_cell_gt5K.pdf")

# Show plot
fig.show()
#fig.update_traces(textfont = dict(size = 7))

/home/xiangru.huo-umw/conda/Jupiter/lib/python3.11/site-packages/plotly/express/_core.py:1992: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [14]:
#selected interphase or ProM cells dataframe
Interphase_cell_df = ShortVSMitotic_df[(ShortVSMitotic_df['Short_ratio(1.5K-2Mb)'] > 0.6) & (ShortVSMitotic_df['Mitotic_ratio(2Mb-12Mb)'] < 0.15)]
ProM_cell_df = ShortVSMitotic_df[(ShortVSMitotic_df['Short_ratio(1.5K-2Mb)'] < 0.65) & (ShortVSMitotic_df['Mitotic_ratio(2Mb-12Mb)'] > 0.25)]
#selected cell with contacts > 30K 
Interphase_cell_30K = Interphase_cell_df[Interphase_cell_df['Num_Contacts'] > 30000]
ProM_cell_30K = ProM_cell_df[ProM_cell_df['Num_Contacts'] > 30000]
#selected cell with contacts > 20K 
Interphase_cell_20K = Interphase_cell_df[Interphase_cell_df['Num_Contacts'] > 20000]
ProM_cell_20K = ProM_cell_df[ProM_cell_df['Num_Contacts'] > 20000]
#selected cell with contacts > 10K 
Interphase_cell_10K = Interphase_cell_df[Interphase_cell_df['Num_Contacts'] > 10000]
ProM_cell_10K = ProM_cell_df[ProM_cell_df['Num_Contacts'] > 10000]
#selected cell with contacts > 5K 
Interphase_cell_5K = Interphase_cell_df[Interphase_cell_df['Num_Contacts'] > 5000]
ProM_cell_5K = ProM_cell_df[ProM_cell_df['Num_Contacts'] > 5000]

ProM_cell_30K

,Short_ratio(1.5K-2Mb),Mitotic_ratio(2Mb-12Mb),Num_Contacts,stage,plate,plate+stage
scHiC_cell46.hg38.nodups.pairs.gz,0.588126,0.255765,52055,H1,plate6_,plate6_H1
scHiC_cell58.hg38.nodups.pairs.gz,0.594854,0.35274,59588,H1,plate6_,plate6_H1
scHiC_cell46.hg38.nodups.pairs.gz,0.588126,0.255765,52055,ES,plate6_,plate6_ES
scHiC_cell58.hg38.nodups.pairs.gz,0.594854,0.35274,59588,ES,plate6_,plate6_ES


In [15]:
len(Interphase_cell_20K[Interphase_cell_20K['plate+stage'] == 'plate8_ES'])

33

In [16]:
len(Interphase_cell_30K[Interphase_cell_30K['plate+stage'] == 'plate8_ES'])

17

In [40]:
ProM_cell_20K

,Short_ratio(1.5K-2Mb),Mitotic_ratio(2Mb-12Mb),Num_Contacts,stage,plate,plate+stage
scHiC_cell46.hg38.nodups.pairs.gz,0.588126,0.255765,52055,ES,plate6_,plate6_ES
scHiC_cell58.hg38.nodups.pairs.gz,0.594854,0.35274,59588,ES,plate6_,plate6_ES


In [58]:
#saving cell info into txt 
##(take care of different batch of mHep/H1 you used!!)

Interphase_cell_df.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_5stages_1.txt", sep='\t', index=True)
ProM_cell_df.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_5stages_1.txt", sep='\t', index=True)
Interphase_cell_10K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_10K_5stages_1.txt", sep='\t', index=True)
ProM_cell_10K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_10K_5stages_1.txt", sep='\t', index=True)
Interphase_cell_5K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_5K_5stages_1.txt", sep='\t', index=True)
ProM_cell_5K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_5K_5stages_1.txt", sep='\t', index=True)
Interphase_cell_20K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_20K_5stages_1.txt", sep='\t', index=True)
ProM_cell_20K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_20K_5stages_1.txt", sep='\t', index=True)
Interphase_cell_30K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_30K_5stages_1.txt", sep='\t', index=True)
ProM_cell_30K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_30K_5stages_1.txt", sep='\t', index=True)

#Interphase_cell_df.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_5stages.txt", sep='\t', index=True)
#ProM_cell_df.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_5stages.txt", sep='\t', index=True)
#Interphase_cell_10K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_10K_5stages.txt", sep='\t', index=True)
#ProM_cell_10K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_10K_5stages.txt", sep='\t', index=True)
#Interphase_cell_5K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_5K_5stages.txt", sep='\t', index=True)
#ProM_cell_5K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_5K_5stages.txt", sep='\t', index=True)
#Interphase_cell_30K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/interphase_cell_30K_5stages.txt", sep='\t', index=True)
#ProM_cell_30K.to_csv("/home/xiangru.huo-umw/scHiC/scripts/inter_file/proM_cell_30K_5stages.txt", sep='\t', index=True)
